In [1]:
import pandas as pd
from pathlib import Path

# ============================================================
# LOAD PHASE 1 DATASET
# ============================================================

DATA_PATH = Path(
    r"C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML"
) / "data" / "ml_ready" / "phase1_event_level_ml_dataset.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (439251, 26)

Columns:
['acq_date', 'lat_cell', 'lon_cell', 'event_number', 'start_time', 'end_time', 'observation_count', 'latitude', 'longitude', 'mean_frp', 'peak_frp', 'total_frp', 'mean_brightness', 'persistent', 'historical_detection_count', 'historical_mean_frp', 'local_frp_deviation', 'historical_daily_activity', 'previously_detected', 'event_date', 'has_historical_context', 'event_id', 'anomaly_prediction', 'anomaly_score', 'is_anomaly', 'anomaly_method']

First 5 rows:


,acq_date,lat_cell,lon_cell,event_number,start_time,end_time,observation_count,latitude,longitude,mean_frp,...,local_frp_deviation,historical_daily_activity,previously_detected,event_date,has_historical_context,event_id,anomaly_prediction,anomaly_score,is_anomaly,anomaly_method
0,2025-01-01,2989,9403,1,2025-01-01 03:50:00,2025-01-01 03:50:00,1,26.9361,94.0335,6.5,...,0.0,0.0,0.0,2025-01-01,1,0,1,0.093408,False,Isolation Forest prototype
1,2025-01-01,2810,9160,1,2025-01-01 03:51:00,2025-01-01 03:51:00,1,25.3231,91.6008,8.0,...,0.0,0.0,0.0,2025-01-01,1,1,1,0.106868,False,Isolation Forest prototype
2,2025-01-01,2811,9159,1,2025-01-01 03:51:00,2025-01-01 03:51:00,1,25.3249,91.5900,11.3,...,0.0,0.0,0.0,2025-01-01,1,2,1,0.096588,False,Isolation Forest prototype
3,2025-01-01,2916,9281,1,2025-01-01 03:51:00,2025-01-01 03:51:00,1,26.2760,92.8154,11.6,...,0.0,0.0,0.0,2025-01-01,1,3,1,0.005439,False,Isolation Forest prototype
4,2025-01-01,2970,9276,1,2025-01-01 03:51:00,2025-01-01 03:51:00,1,26.7587,92.7613,5.3,...,0.0,0.0,0.0,2025-01-01,1,4,1,0.108285,False,Isolation Forest prototype


In [2]:
import pandas as pd
from pathlib import Path

# ============================================================
# LOAD PHASE 1 DATA
# ============================================================

BASE_DIR = Path(
    r"C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML"
)

phase1_path = (
    BASE_DIR
    / "data"
    / "ml_ready"
    / "phase1_event_level_ml_dataset.csv"
)

df = pd.read_csv(phase1_path)

print("Phase 1:", df.shape)


# ============================================================
# LOAD OSM-LABELED DATASET
# ============================================================

# Find the file containing source_candidate
search_dir = BASE_DIR / "data"

matches = list(search_dir.rglob("*.csv"))

print("\nCSV files containing 'source_candidate':")

for file in matches:
    try:
        temp = pd.read_csv(file, nrows=5)

        if "source_candidate" in temp.columns:
            print(file)

    except Exception:
        pass

Phase 1: (439251, 26)

CSV files containing 'source_candidate':
C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\data\ml_ready\source_label_candidates_sample.csv
C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\data\ml_ready\supervised_source_candidates.csv


In [3]:
# ============================================================
# LOAD OSM SOURCE CANDIDATE LABELS
# ============================================================

labels_path = (
    BASE_DIR
    / "data"
    / "ml_ready"
    / "supervised_source_candidates.csv"
)

labels_df = pd.read_csv(labels_path)

print("Labels dataset shape:", labels_df.shape)

print("\nLabel columns:")
print(labels_df.columns.tolist())

print("\nSource candidate distribution:")
print(labels_df["source_candidate"].value_counts(dropna=False))


# ============================================================
# MERGE WITH PHASE 1
# ============================================================

model_df = df.merge(
    labels_df[["event_id", "source_candidate"]],
    on="event_id",
    how="inner"
)

print("\nPrototype dataset shape:", model_df.shape)

print("\nFinal label distribution:")
print(model_df["source_candidate"].value_counts())


# ============================================================
# CHECK DUPLICATES
# ============================================================

print(
    "\nDuplicate event IDs:",
    model_df["event_id"].duplicated().sum()
)

print(
    "\nMissing labels:",
    model_df["source_candidate"].isna().sum()
)

Labels dataset shape: (28790, 10)

Label columns:
['event_id', 'observation_count', 'mean_frp', 'peak_frp', 'historical_detection_count', 'historical_mean_frp', 'local_frp_deviation', 'previously_detected', 'anomaly_score', 'source_candidate']

Source candidate distribution:
source_candidate
Forest_Natural         24213
Industrial              3758
Agriculture_Biomass      442
Waste_Other              377
Name: count, dtype: int64

Prototype dataset shape: (28790, 27)

Final label distribution:
source_candidate
Forest_Natural         24213
Industrial              3758
Agriculture_Biomass      442
Waste_Other              377
Name: count, dtype: int64

Duplicate event IDs: 0

Missing labels: 0


In [4]:
from sklearn.model_selection import GroupShuffleSplit

# ============================================================
# PROTOTYPE FEATURES
# ============================================================

features = [
    "observation_count",
    "mean_frp",
    "peak_frp",
    "total_frp",
    "mean_brightness",
    "persistent",
    "historical_detection_count",
    "historical_mean_frp",
    "local_frp_deviation",
    "historical_daily_activity",
    "previously_detected",
    "anomaly_score",
    "is_anomaly"
]

target = "source_candidate"


# ============================================================
# CREATE MODEL DATA
# ============================================================

X = model_df[features].copy()
y = model_df[target].copy()


# Make sure numeric
for col in features:
    X[col] = pd.to_numeric(X[col], errors="coerce")

X = X.replace([float("inf"), float("-inf")], 0)
X = X.fillna(0)


# ============================================================
# SPATIAL GROUP
# ============================================================

GRID_SIZE = 0.1

model_df["spatial_group"] = (
    (model_df["latitude"] / GRID_SIZE)
    .apply(lambda x: int(x))
    .astype(str)
    + "_"
    +
    (model_df["longitude"] / GRID_SIZE)
    .apply(lambda x: int(x))
    .astype(str)
)


# ============================================================
# TRAIN / TEMP SPLIT
# ============================================================

gss1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    gss1.split(
        X,
        y,
        groups=model_df["spatial_group"]
    )
)


X_train = X.iloc[train_idx].copy()
y_train = y.iloc[train_idx].copy()

X_temp = X.iloc[temp_idx].copy()
y_temp = y.iloc[temp_idx].copy()

groups_temp = model_df.iloc[temp_idx]["spatial_group"]


# ============================================================
# VALIDATION / TEST SPLIT
# ============================================================

gss2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    gss2.split(
        X_temp,
        y_temp,
        groups=groups_temp
    )
)

X_val = X_temp.iloc[val_idx].copy()
y_val = y_temp.iloc[val_idx].copy()

X_test = X_temp.iloc[test_idx].copy()
y_test = y_temp.iloc[test_idx].copy()


# ============================================================
# RESULTS
# ============================================================

print("TRAIN:", X_train.shape)
print("VALIDATION:", X_val.shape)
print("TEST:", X_test.shape)

print("\nTrain labels:")
print(y_train.value_counts())

print("\nValidation labels:")
print(y_val.value_counts())

print("\nTest labels:")
print(y_test.value_counts())

TRAIN: (20340, 13)
VALIDATION: (4073, 13)
TEST: (4377, 13)

Train labels:
source_candidate
Forest_Natural         16459
Industrial              3242
Agriculture_Biomass      354
Waste_Other              285
Name: count, dtype: int64

Validation labels:
source_candidate
Forest_Natural         3821
Industrial              203
Agriculture_Biomass      29
Waste_Other              20
Name: count, dtype: int64

Test labels:
source_candidate
Forest_Natural         3933
Industrial              313
Waste_Other              72
Agriculture_Biomass      59
Name: count, dtype: int64


### Train LightGBM

In [5]:
import numpy as np
import lightgbm as lgb

from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix
)

# ============================================================
# CLASS WEIGHTS
# ============================================================

class_counts = y_train.value_counts()

class_weights = {
    cls: len(y_train) / (len(class_counts) * count)
    for cls, count in class_counts.items()
}

print("Class weights:")
for cls, weight in class_weights.items():
    print(f"{cls}: {weight:.2f}")


# ============================================================
# SAMPLE WEIGHTS
# ============================================================

sample_weights = y_train.map(class_weights)


# ============================================================
# LABEL ENCODING
# ============================================================

classes = sorted(y_train.unique())

label_to_int = {
    label: i
    for i, label in enumerate(classes)
}

int_to_label = {
    i: label
    for label, i in label_to_int.items()
}

y_train_encoded = y_train.map(label_to_int)
y_val_encoded = y_val.map(label_to_int)
y_test_encoded = y_test.map(label_to_int)


print("\nClasses:")
print(classes)


# ============================================================
# LIGHTGBM MODEL
# ============================================================

model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=len(classes),

    n_estimators=300,
    learning_rate=0.05,

    num_leaves=31,
    max_depth=-1,

    subsample=0.8,
    colsample_bytree=0.8,

    random_state=42,
    n_jobs=-1
)


# ============================================================
# TRAIN
# ============================================================

model.fit(
    X_train,
    y_train_encoded,
    sample_weight=sample_weights,

    eval_set=[
        (X_val, y_val_encoded)
    ],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=30,
            verbose=True
        )
    ]
)


print("\nTraining completed.")
print("Best iteration:", model.best_iteration_)

Class weights:
Forest_Natural: 0.31
Industrial: 1.57
Agriculture_Biomass: 14.36
Waste_Other: 17.84

Classes:
['Agriculture_Biomass', 'Forest_Natural', 'Industrial', 'Waste_Other']
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002158 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2309
[LightGBM] [Info] Number of data points in the train set: 20340, number of used features: 13
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
Training until validation scores don't improve for 30 rounds


c:\Users\DIPANSHU SHUKLA\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Did not meet early stopping. Best iteration is:
[300]	valid_0's multi_logloss: 0.400137

Training completed.
Best iteration: 300


In [6]:
from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score,
    f1_score,
    accuracy_score,
    confusion_matrix
)
import pandas as pd


# ============================================================
# TEST PREDICTION
# ============================================================

y_pred_encoded = model.predict(X_test)

# Convert numeric predictions back to class names
y_pred = pd.Series(y_pred_encoded).map(int_to_label).values


# ============================================================
# METRICS
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

balanced_acc = balanced_accuracy_score(
    y_test,
    y_pred
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)


print("=" * 60)
print("PROTOTYPE LIGHTGBM TEST RESULTS")
print("=" * 60)

print(f"Accuracy          : {accuracy:.4f}")
print(f"Balanced Accuracy : {balanced_acc:.4f}")
print(f"Macro F1          : {macro_f1:.4f}")
print(f"Weighted F1       : {weighted_f1:.4f}")


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=classes,
        target_names=classes,
        zero_division=0
    )
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=classes
)

cm_df = pd.DataFrame(
    cm,
    index=classes,
    columns=classes
)

print("\nConfusion Matrix:")
display(cm_df)

PROTOTYPE LIGHTGBM TEST RESULTS
Accuracy          : 0.8659
Balanced Accuracy : 0.4491
Macro F1          : 0.4400
Weighted F1       : 0.8879

Classification Report:
                     precision    recall  f1-score   support

Agriculture_Biomass       0.03      0.15      0.05        59
     Forest_Natural       0.96      0.91      0.93      3933
         Industrial       0.66      0.62      0.64       313
        Waste_Other       0.17      0.11      0.13        72

           accuracy                           0.87      4377
          macro avg       0.46      0.45      0.44      4377
       weighted avg       0.91      0.87      0.89      4377


Confusion Matrix:


,Agriculture_Biomass,Forest_Natural,Industrial,Waste_Other
Agriculture_Biomass,9,49,1,0
Forest_Natural,285,3578,47,23
Industrial,10,92,195,16
Waste_Other,2,11,51,8


In [7]:
import joblib
import json
from pathlib import Path

# ============================================================
# MODEL DIRECTORY
# ============================================================

MODEL_DIR = BASE_DIR / "models"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# SAVE LIGHTGBM MODEL
# ============================================================

model_path = MODEL_DIR / "lightgbm_source_classifier.pkl"

joblib.dump(
    model,
    model_path
)


# ============================================================
# SAVE FEATURE LIST
# ============================================================

feature_path = MODEL_DIR / "prototype_features.json"

with open(
    feature_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        features,
        f,
        indent=4
    )


# ============================================================
# SAVE LABEL MAPPING
# ============================================================

label_path = MODEL_DIR / "label_mapping.json"

with open(
    label_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "label_to_int": label_to_int,
            "int_to_label": {
                str(k): v
                for k, v in int_to_label.items()
            }
        },
        f,
        indent=4
    )


print("Model saved:", model_path)
print("Features saved:", feature_path)
print("Labels saved:", label_path)

Model saved: C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\models\lightgbm_source_classifier.pkl
Features saved: C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\models\prototype_features.json
Labels saved: C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\models\label_mapping.json
